In [0]:
CDC_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/accounts/account_cdc.csv"

cdc_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CDC_PATH)
)

display(cdc_df)

In [0]:
print("Rows:", cdc_df.count())

cdc_df.printSchema()

In [0]:
from pyspark.sql import functions as F

print("Total rows:", cdc_df.count())

print("\nChange types:")
display(
    cdc_df.groupBy("change_type")
           .count()
           .orderBy(F.col("count").desc())
)

print("\nChanged columns:")
display(
    cdc_df.groupBy("changed_column")
           .count()
           .orderBy(F.col("count").desc())
)

print("\nNull counts:")
display(
    cdc_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in cdc_df.columns
    ])
)

print("\nDuplicate CDC rows:")
duplicate_count = (
    cdc_df.groupBy(cdc_df.columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
)

print(duplicate_count)

print("\nTimestamp range:")
display(
    cdc_df.select(
        F.min("change_timestamp").alias("min_change_timestamp"),
        F.max("change_timestamp").alias("max_change_timestamp")
    )
)

In [0]:
from pyspark.sql import functions as F

CDC_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/accounts/account_cdc.csv"

BRONZE_TABLE = "dbx_fintech_data_platform.bronze.account_cdc"

PIPELINE_NAME = "account_cdc_ingestion"

TARGET_TABLE = BRONZE_TABLE

In [0]:
cdc_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CDC_PATH)
)

print(f"Source records: {cdc_df.count()}")

cdc_df.printSchema()

In [0]:
bronze_cdc_df = (
    cdc_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(CDC_PATH))
)

In [0]:
display(bronze_cdc_df)

In [0]:
print("Bronze records:", bronze_cdc_df.count())

In [0]:
already_processed = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM dbx_fintech_data_platform.metadata.ingestion_log
    WHERE pipeline_name = '{PIPELINE_NAME}'
      AND source_file = '{CDC_PATH}'
      AND status = 'SUCCESS'
""").collect()[0]["cnt"]

print(f"Already processed successfully: {already_processed}")

In [0]:
if already_processed == 0:

    start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

    try:

        row_count = bronze_cdc_df.count()

        (
            bronze_cdc_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )

        end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

        print(f"Successfully ingested {row_count} CDC records.")

        spark.sql(f"""
            INSERT INTO dbx_fintech_data_platform.metadata.ingestion_log
            VALUES (
                '{PIPELINE_NAME}',
                '{CDC_PATH}',
                NULL,
                '{TARGET_TABLE}',
                'SUCCESS',
                {row_count},
                TIMESTAMP('{start_time}'),
                TIMESTAMP('{end_time}'),
                NULL
            )
        """)

        print("Ingestion audit recorded.")

    except Exception as e:

        error_message = str(e).replace("'", "''")

        end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

        spark.sql(f"""
            INSERT INTO dbx_fintech_data_platform.metadata.ingestion_log
            VALUES (
                '{PIPELINE_NAME}',
                '{CDC_PATH}',
                NULL,
                '{TARGET_TABLE}',
                'FAILED',
                0,
                TIMESTAMP('{start_time}'),
                TIMESTAMP('{end_time}'),
                '{error_message}'
            )
        """)

        raise

else:

    print("Source file already processed successfully.")
    print("Skipping ingestion to maintain idempotency.")

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT account_id) AS unique_accounts,
    COUNT(DISTINCT _source_file) AS source_files,
    MIN(change_timestamp) AS min_change_timestamp,
    MAX(change_timestamp) AS max_change_timestamp
FROM dbx_fintech_data_platform.bronze.account_cdc;

In [0]:
%sql

SELECT
    change_type,
    COUNT(*) AS records
FROM dbx_fintech_data_platform.bronze.account_cdc
GROUP BY change_type;

In [0]:
%sql

SELECT
    changed_column,
    COUNT(*) AS records
FROM dbx_fintech_data_platform.bronze.account_cdc
GROUP BY changed_column
ORDER BY records DESC;

In [0]:
%sql

SELECT
    account_id,
    change_type,
    changed_column,
    old_value,
    new_value,
    change_timestamp
FROM dbx_fintech_data_platform.bronze.account_cdc
ORDER BY change_timestamp
LIMIT 20;

In [0]:
%sql

SELECT
    pipeline_name,
    source_file,
    target_table,
    status,
    rows_processed,
    started_at,
    completed_at,
    error_message
FROM dbx_fintech_data_platform.metadata.ingestion_log
WHERE pipeline_name = 'account_cdc_ingestion'
ORDER BY completed_at DESC;

In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM dbx_fintech_data_platform.bronze.account_cdc;